> ## 📘 Fundamentals required for M1L2 - Process Multimodal Data
> Before working this notebook, make sure you're comfortable with these — every cell below assumes them:
>
- **Everything from M1L1** — this lab reuses the WatsonX call pattern, JSON load/save, and prompt-building from Lesson 1 -- if that felt shaky, revisit it first.
- **Images in Python (PIL)** — <code>PIL.Image.open(path)</code> loads an image file into an object you can display (<code>plt.imshow</code>) or process.
- **Vision-language (multimodal) models** — A model that accepts BOTH an image and text in one prompt and returns a text description -- unlike the text-only model in M1L1. The message format adds an <code>image_url</code> content block alongside the normal text block.
- **Base64 image encoding** — LLM APIs can't accept a raw image file directly in JSON -- the image bytes are base64-encoded into a text string first (<code>data:image/jpeg;base64,...</code>), which is how you embed binary data inside a text-based API request.
- **HTTP requests (the <code>requests</code> library)** — <code>requests.get(url)</code> downloads a file (here, review photos) from a remote URL; <code>.content</code> gives you the raw bytes to save locally.
- **<code>ast.literal_eval</code>** — The review dataset stores a list of image URLs AS A STRING (e.g. <code>"['url1','url2']"</code>). <code>ast.literal_eval()</code> safely converts that string back into a real Python list -- a common need when data comes from CSV/JSON exports.
- **Retry with exponential backoff** — The <code>@retry</code> decorator (from <code>tenacity</code>) automatically re-attempts a flaky network call, waiting longer between each failure (1s, 2s, 4s...) -- essential when downloading many images where some requests will randomly fail.
- **zipfile extraction** — <code>zipfile.ZipFile(...).extractall()</code> unpacks a downloaded <code>.zip</code> archive (here, the recipe images) into the working directory.
>
> If any of these are new, it's worth a 10-minute detour before continuing — the cell-by-cell notes
> below assume you already know what each one IS, and focus on how this lab USES it.

<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Process Multimodal Data with LLMs**


Estimated time needed: **45** minutes


## **Scenario**


### **Background**

You transformed unstructured restaurant descriptions into a structured JSON knowledge base using LLMs. This allowed the app to reason over textual attributes such as cuisine type, pricing, and signature dishes in a consistent, machine-readable format.

However, the app’s data is not limited to text. It also includes **food recipes** and **user visit histories** that reference **images**, capturing visual information that text alone cannot fully describe, such as presentation style, ingredients, and portion details. To unlock this information and integrate it with the original databases, you will use multimodal GenAI capabilities to convert images into descriptive text and enrich the structured knowledge you have.


### **The challenge**

Images are inherently unstructured and cannot be directly stored or queried in a text-based knowledge system. To make them usable, you must first **generate high-quality textual descriptions** that summarize the visual content of each image.

Your challenge in this lab is to:

* Apply image captioning to generate descriptive text for food images
* Align and merge image captions with the corresponding recipe and restaurant JSON records
* Extend the existing structured knowledge base with multimodal context while preserving a consistent schema

By the end of this lab, you will have enriched your restaurant knowledge base with visual insights. You will bridge text and images into a unified, machine-accessible representation, setting the stage for more intelligent multimodal reasoning in later assignments.


## **Objectives**

In this lab, you will write a Python program that will:

* Load food recipe and user visit data containing image references
* Use a multimodal foundation model to generate textual captions for food images
* Integrate the generated image captions into the existing structured JSON data
* Produce an enriched multimodal JSON knowledge file for downstream applications


## **Important: About the lab environment**


Please be aware that sessions for this lab environment are not persisted. Every time you connect to this lab, a new environment is created for you. Any data you may have saved in the earlier session would get lost. Plan to complete these labs in a single session, to avoid losing your data.


## **Screenshot requirement for this lab**


You will be prompted to take a screenshot and save it on your own device. You will need this screenshot either to answer graded quiz questions or to upload as your submission for the Final Project at the end of this course. You can use various free screen-grabbing tools or your operating system's shortcut keys to do this (for example, `Alt+PrintScreen` on Windows and `Command+shift+4` on Mac).
**Note**: The screenshot can be saved with either the **.jpg** or **.png** extension.


----


## **Set up the lab environment**


For this lab, you will still be using the following libraries:

* [`numpy`](https://numpy.org/) for numerical operations and handling array-based data during preprocessing and analysis

* [`matplotlib`](https://matplotlib.org/) for basic data visualization and plotting, useful for inspecting distributions or intermediate results

* [`json`](https://docs.python.org/3/library/json.html) for parsing, constructing, and serializing structured JSON representations extracted from unstructured text

* **IBM watsonx AI SDK** (`ibm-watsonx-ai`) for interacting with foundation models hosted on IBM watsonx.ai:

  * `Credentials` to securely authenticate with the watsonx.ai service
  * `ModelInference` to invoke foundation models for text understanding and information extraction
  * `GenTextParamsMetaNames` to configure text generation and extraction parameters
  * `ModelTypes` and `DecodingMethods` to select the appropriate model and control inference behavior

These libraries together enable you to transform images into text-based data using GenAI-powered workflows.


### Installing required libraries

Run the following code block to install all required libraries:


> 📎 **CONCEPT** — Installs the pinned libraries for this lab (numpy, matplotlib, the WatsonX SDK) -- same pattern as M1L1's install cell.

In [ ]:
%%capture
%pip install numpy==2.3.4
%pip install matplotlib==3.10.7
%pip install ibm-watsonx-ai==1.4.7

### Importing required libraries

It is recommended to import all required libraries in one place (here):


> 📎 **CONCEPT** — Imports needed for image handling (<code>PIL.Image</code>) plus the same WatsonX classes and warning-suppression setup from M1L1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from PIL import Image

# IBM WatsonX imports
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)

# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Fetch the data file

Run the following code to fetch the food recipe, user visit history, and recipe images.


> 📎 **CONCEPT** — Downloads three files from IBM's course bucket with <code>wget</code>: the recipe metadata JSON, the synthetic user-review JSON, and a zip of recipe images -- the raw inputs for this whole lab.

In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hpTjb6liKBLVHQK0UgMi5A/Recipes.json
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fQUs9wQ6aB6ts6fmkD2V2w/Synthetic-User-Reviews.json
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5_Rr6ohviItzucyWk6nkrw/synthetic-recipe-images.zip

Run the following code to unzip the image folder:


> 📎 **CONCEPT** — Unzips <code>synthetic-recipe-images.zip</code> into the working directory so the image files referenced by the recipe JSON actually exist on disk.

In [ ]:
import zipfile

with zipfile.ZipFile("synthetic-recipe-images.zip", 'r') as zip_ref:
    zip_ref.extractall()

----


## **Exercise 1: Preprocess the food reciepe data**


### Step 1: Explore the food recipe JSON File and its images

Before applying any multimodal processing, it is important to understand the structure and contents of the data you will be working with. In this step, you will load the food recipe JSON file and examine its fields, including textual metadata and image references.

You will also inspect a sample of the associated food images to understand the visual information they contain, such as ingredients, presentation style, and portion size. This exploration step helps you identify how the image data aligns with recipe metadata and prepares you for integrating image captions into the existing JSON structure in later steps.

By the end of this step, you should have a clear understanding of how the recipe data and images are organized, and how they relate to the structured restaurant knowledge built in the previous lesson.


> 📎 **CONCEPT** — <b>Solved exercise 1.1-1.3.</b> Loads <code>Recipes.json</code> into <code>recipe_data</code>, prints every key/value/type pair of the first recipe so you can see its shape, then opens and displays that recipe's image with matplotlib -- confirming both the JSON and image data loaded correctly.

In [ ]:
### Step 1.1: Load the json file. Define the loaded data as recipe_data.
with open('Recipes.json', 'r', encoding='utf-8') as f:
    recipe_data = json.load(f)

### Step 1.2: Print each key-value pair of the first recipe. In this format: key (type of value): value
first_recipe = recipe_data[0]
for key, value in first_recipe.items():
    print(f"{key} ({type(value).__name__}): {value}")

### Step 1.3: Show the image of the first recipe (recipe1)
img_path = first_recipe['image_path']
img = Image.open(img_path)
plt.imshow(img)
plt.axis('off')
plt.title(first_recipe.get('name', 'Recipe 1'))
plt.show()


### Step 2: Define the vision LLM with LLaMA

To generate meaningful textual descriptions from food images, you will use a vision-capable large language model (LLM). In this step, you will define and configure a **LLaMA-based vision model** that can process both images and text prompts.

You will initialize the model with the appropriate credentials and inference parameters, and verify that it accepts image inputs and produces descriptive captions. This model will serve as the core component for converting visual information into natural language that can be integrated with your existing JSON-based knowledge structure.

By completing this step, you will have a vision-enabled LLM ready to perform image captioning reliably and repeatably.


> 📎 **CONCEPT** — Defines <code>vision_llm()</code> -- the multimodal counterpart to M1L1's <code>llm_model()</code>. Uses a vision-capable model (<code>llama-4-maverick</code>), base64-encodes the input image, and builds a message where the user content is a LIST containing both an <code>image_url</code> block and a <code>text</code> block -- this is the standard multimodal message shape.

In [ ]:
import base64

def vision_llm(system_msg, prompt_txt, image_path):
    #system_msg: input system message for the LLM
    #prompt_txt: input user prompt for the LLM
    #image_path: the file path of the input image

    ### Credentials of the model
    model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8'
    project_id = "skills-network"
    credentials = Credentials(
                    url="https://us-south.ml.cloud.ibm.com",
                    )
    generate_params = {"max_tokens": 300}

    ### Step 2.1: Define the model by ModelInference
    model = ModelInference(
        model_id=model_id,
        credentials=credentials,
        project_id=project_id,
        params=generate_params,
    )

    ### Step 2.2: Encode the input image to a base64 string
    with open(image_path, 'rb') as img_file:
        image_b64 = base64.b64encode(img_file.read()).decode('utf-8')

    ### Step 2.3: Define the messages for the model
    messages = [
        {
            "role": "system",
            "content": system_msg,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_b64}"
                    },
                },
                {
                    "type": "text",
                    "text": prompt_txt,
                },
            ],
        },
    ]

    ### Step 2.4: Get the response for the messages
    response = model.chat(messages=messages)
    return response['choices'][0]['message']['content']


### Step 3: Design and validate prompts for the vision LLM

To ensure the vision LLM generates useful and consistent image captions, careful prompt design is essential. In this step, you will create a prompt that instructs the model to focus on relevant visual details, such as ingredients, cooking style, and presentation, while avoiding unnecessary or speculative information.

Once the prompt is defined, you will perform a simple unit test by running the model on a small sample of food images. This allows you to verify that the generated captions are accurate, concise, and suitable for integration into the existing JSON data structure.

By the end of this step, you will have a validated prompt that reliably converts food images into high-quality textual descriptions.


> 📎 **CONCEPT** — Defines <code>image_caption_prompt_template()</code> -- builds a system+user prompt asking the model to describe a food image factually (ingredients, presentation, colors) given just the dish's name for context. Tests it once on the first recipe's image and prints the caption.

In [ ]:
### Define the food image caption prompts given a food name.
### The food name, as you have noticed, comes from the corresponding recipe data.
### You want to include the food name to ensure the model focuses on it while giving captions.
def image_caption_prompt_template(food_name):
    # food_name: the food name of the recipe

    ### Step 3.1: Design the prompts
    image_caption_system_msg = (
        "You are a culinary expert and food photographer with deep knowledge of ingredients, "
        "cooking techniques, and food presentation. Your task is to generate concise, accurate, "
        "and informative descriptions of food images."
    )
    image_caption_prompt_txt = (
        f"This is an image of '{food_name}'. Describe the dish in 2-3 sentences, focusing on "
        "the visible ingredients, cooking style, presentation, colors, and portion size. "
        "Be specific and factual. Do not speculate about taste or smell."
    )

    return image_caption_system_msg, image_caption_prompt_txt


### Test the prompts on the first recipe
### Step 3.2: Get the prompts with the food name of the first recipe
first_food_name = recipe_data[0].get('name', 'Food')
system_msg, prompt_txt = image_caption_prompt_template(first_food_name)

### Step 3.3: Get the test response and print it
response = vision_llm(system_msg, prompt_txt, recipe_data[0]['image_path'])
print(response)


### Step 4: Caption all images and augment the data

With a validated prompt and a working vision LLM, you will now scale the image captioning process across the entire dataset. In this step, you will iterate through all food images, generate captions for each one, and associate the results with their corresponding recipe or restaurant records.

You will then augment the existing JSON data by adding the generated image captions as new fields, ensuring the final structure remains consistent and machine-readable. This enriched dataset combines textual and visual insights into a unified representation, extending the structured knowledge base created in the previous lessons.

By the end of this step, you will have a multimodal JSON file that integrates image-derived descriptions with existing structured data, ready for downstream GenAI applications.


> 📎 **CONCEPT** — <b>The main captioning loop.</b> Runs <code>vision_llm()</code> over EVERY recipe image (can take up to 15 minutes), storing each generated caption back into <code>recipe_data[i]['image_description']</code> -- this is what turns a folder of images into searchable TEXT, the exact idea the Chroma Studio guide called out as the bridge to multimodal RAG.

In [ ]:
### Get captions for each image in the dataset and add to the JSON file (up to 15 minutes)
### recipe_data is the recipe you loaded in Step 1
for i in range(len(recipe_data)):
    if (i+1) % 20 == 0:
        print(f'{i+1} out of {len(recipe_data)} is done')

    ### Step 4.1: Get the caption prompts
    food_name = recipe_data[i].get('name', 'Food')
    system_msg, prompt_txt = image_caption_prompt_template(food_name)

    ### Step 4.2: Get the response with the prompts
    response = vision_llm(system_msg, prompt_txt, recipe_data[i]['image_path'])

    ### Save the response as another item in the recipe data
    recipe_data[i]['image_description'] = response

print('ALL DONE!')


Take a screenshot of the Python code, clearly showing your implementation and the output from Step 2. Name the screenshot ```M1L2_caption_all_recipes.jpg```. 


### Step 5: Save the image-caption-augmented recipe data

Save your data by running the following code.


> 📎 **CONCEPT** — Saves the recipe data (now including the generated captions) to <code>augmented_food_recipe.json</code> -- this is the file M2L1 loads to build the image side of the vector index.

In [ ]:
filename = 'augmented_food_recipe.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(recipe_data, f, indent=4)

You can download the saved JSON file to your local folder. Be sure to store it in a safe location, as you will need it for future assignments.


----


## **Exercise 2: Preprocess the user visit history**

In this exercise, you will largely replicate the workflow from **Exercise 1**. The core steps, data exploration, prompt design, model inference, and JSON augmentation, remain the same.

The key difference lies in **prompt design**. Instead of focusing on food images, you will tailor your prompt to generate concise, informative descriptions from user visit history data, such as contextual cues from URLs or associated images combined with their written reviews. These prompts should emphasize elements that help explain user preferences and behavior while maintaining consistency with the existing JSON schema.

By the end of this exercise, you will have enriched the user visit history data with structured, machine-readable descriptions, further expanding the multimodal knowledge base built throughout this lab.


### Step 1: Load the user review data

Similar to Step 1 of Exercise 1, complete the following code block to load and explore the user review data.


> 📎 **CONCEPT** — <b>Solved exercise 1.1-1.2 (Part 1).</b> Loads <code>Synthetic-User-Reviews.json</code> into <code>user_review_data</code> and prints the first review's fields -- same load-and-inspect pattern as before, now on a different dataset.

In [ ]:
### Part 1
### Step 1.1: Load the review dataset with variable name user_review_data
with open('Synthetic-User-Reviews.json', 'r', encoding='utf-8') as f:
    user_review_data = json.load(f)

### Step 1.2: Print the first review by key-value pairs
first_review = user_review_data[0]
for key, value in first_review.items():
    print(f"{key} ({type(value).__name__}): {value}")


Let's look at the 'images' item. Although it looks like a list, it is actually a list in string type. Therefore, you need to be careful and convert the string to the actual Python list. Complete the code block below to display the first review image of the first restaurant visit.


> 📎 **CONCEPT** — <b>Solved exercise 1.3-1.6 (Part 2).</b> The review's <code>images</code> field is a STRINGIFIED list (see the <code>ast.literal_eval</code> fundamental above) -- converts it to a real list, downloads the first image with <code>requests.get()</code>, saves it locally, then opens and displays it.

In [ ]:
### Part 2
### Get the image by requesting from the URL
import ast  # needed to convert string representation of list to actual python list
import requests

### Step 1.3: Use ast.literal_eval to convert the string list of images to an actual Python list
first_review_images = ast.literal_eval(user_review_data[0]['images'])

### Step 1.4: Use the requests.get() method to get the image content
first_img_url = first_review_images[0]
img_response = requests.get(first_img_url)

### Step 1.5: Write the image content in Step 1.4 to a temporary file 'review_image_placeholder.jpg'
with open('review_image_placeholder.jpg', 'wb') as img_file:
    img_file.write(img_response.content)

### Step 1.6: Open the 'review_image_placeholder.jpg', and show the image
img = Image.open('review_image_placeholder.jpg')
plt.imshow(img)
plt.axis('off')
plt.title('User Review Image')
plt.show()


### Step 2: Define the prompt and the validation

Again, you will design a prompt template for the vision task and perform a unit test to validate the model’s behavior. However, unlike the previous exercise, this prompt must **incorporate textual context from user reviews** to guide the image captioning process.

In this step, you will define a prompt template function that takes user review text as input and uses it as contextual information when describing a food image. The system message assigns the model the role of a culinary expert, while the prompt instructs the model to generate a concise image description that aligns with the sentiment and details expressed in the reviews.

After defining the prompt template, you will perform a unit test by running the vision LLM on a single image and its associated reviews. This test allows you to verify that the generated caption effectively combines visual information from the image with contextual cues from the reviews before applying the prompt at scale in later steps.


> 📎 **CONCEPT** — Defines <code>review_context_image_caption_prompt_template()</code> -- similar to the recipe captioner, but this prompt ALSO includes the written review text as context, so the caption reflects what the reviewer actually said, not just what's visible. Tests it once and prints the result.

In [ ]:
### Prompt template: caption the images with the context of the reviews
def review_context_image_caption_prompt_template(reviews):
    # reviews: the written review content

    ### Step 2.1: Design your prompts
    review_context_image_caption_system_msg = (
        "You are a culinary expert analyzing food images in the context of restaurant reviews. "
        "Your goal is to generate concise, factual image descriptions that complement and align "
        "with the sentiment and details expressed in user reviews."
    )
    review_context_image_caption_prompt_txt = (
        f"A user visited a restaurant and left the following review:\n\n\"{reviews}\"\n\n"
        "Based on the image shown and the context of this review, describe the food or dining "
        "experience visible in the image in 2-3 sentences. Focus on visible ingredients, "
        "presentation, portion size, and any visual cues that relate to the review."
    )

    return review_context_image_caption_system_msg, review_context_image_caption_prompt_txt


### Step 2.2: Get the prompts
first_review_text = user_review_data[0].get('review', '')
system_msg, prompt_txt = review_context_image_caption_prompt_template(first_review_text)

### Step 2.3: Get the response by the vision_llm you defined previously
response = vision_llm(system_msg, prompt_txt, 'review_image_placeholder.jpg')
print(response)


### Step 3: Caption all images and augment the data

Now, you are ready to get all captions with a for loop! Complete and run the following code block to caption all the review images.


> 📎 **CONCEPT** — <b>The main review-captioning loop with retry.</b> <code>get_data_with_retry()</code> wraps the image download in the <code>@retry</code> decorator so a single flaky request doesn't kill the whole loop. For every review, downloads each of its images, generates a review-aware caption, and collects them into <code>review_image_captions</code> -- appended to each review as <code>image_captions</code>.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

### URL Request function with Retry
# Retries up to 10 times, starting at 1s and doubling (1s, 2s, 4s...)
@retry(stop=stop_after_attempt(10), wait=wait_exponential(multiplier=1, min=1, max=10))
def get_data_with_retry(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status()  # Must raise error for retry to trigger
    return response

### Start the for loop
for i in range(len(user_review_data)):
    ### Step 3.1: Convert the string to the Python list of image urls
    review_images = ast.literal_eval(user_review_data[i]['images'])

    review_image_captions = []
    if len(review_images) > 0:
        for img_url in review_images:
            try:
                ### Step 3.2: Use get_data_with_retry to get the image_data
                image_data = get_data_with_retry(img_url)
                print("Success!")
            except Exception as e:
                print(f"All retries failed at url {img_url}:", e)
                continue
            image = image_data.content
            with open('review_image_placeholder.jpg', 'wb') as img_file:
                img_file.write(image)

            ### Step 3.3: Get the prompts, get the response, and append the response to review_image_captions
            review_text = user_review_data[i].get('review', '')
            sys_msg, pmt_txt = review_context_image_caption_prompt_template(review_text)
            caption = vision_llm(sys_msg, pmt_txt, 'review_image_placeholder.jpg')
            review_image_captions.append(caption)

    ### Append the review_image_captions to the review data
    user_review_data[i]['image_captions'] = review_image_captions

print('ALL DONE!')


### Step 4: Save the image-caption-augmented user review data


Run the following code block to save the augmented review data:


> 📎 **CONCEPT** — Saves the augmented review data (now including image captions) to <code>augmented_user_review.json</code> -- the second output file this lab produces.

In [ ]:
filename = 'augmented_user_review.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(user_review_data, f, indent=4)

You can download the saved JSON file to your local folder. Be sure to store it in a safe location, as you will need it for future assignments.


----


## **Conclusion**

You have successfully applied GenAI tools to transform the unstructured text data into a well-structured JSON file!

----


## Authors


[Jianping (Mike) Ye](https://www.linkedin.com/in/jianping-ye/)


<!--## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-02-03|1|Jianping Ye|Create lab|
|2026-02-10|2|Jojy John|ID Reviewed|-->



©IBM Corporation. All rights reserved.
